# Period 2 from $V_{1}(\sigma) = 2 V_{0}(\sigma) - V_{-1}(\sigma)$

In [15]:
import sys, io
import math
import numpy as np
import pandas as pd
from scipy.optimize import nnls
from fractions import Fraction
from sympy import factorint
from itertools import product, combinations
from typing import List, Optional, Tuple
import random
from scipy.sparse import coo_array
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
import pandas as pd

from kenglish.mrlattice import (
    collatzPath,
    N_i, F_0, F_1, 
    mrTupToLaTex, mrTupFromPath, mrTupValue, mrTupRootedValue,
    generationRootedIntegersForwardWithMeta,
    generationIntegersForwardWithMeta, 
    countZeros, 
    generationAffineParamsFromPath,
#    affineFunctionParamsFromPrefix,
)

In [3]:
T = N_i
current_bit = "1"
for i in range(5):
    val_1 = mrTupRootedValue(1, T)
    val_0 = mrTupRootedValue(0, T)
    val_n1 = mrTupRootedValue(-1, T)

    if current_bit == "0":
        status_0 = "not"
        status_1 = "fixed"
    else:
        status_0 = "fixed"
        status_1 = "not"
    
    print(f"{val_1} = 2*{val_0} - {val_n1}   {status_0}:{status_1}")
    if i & 1 == 0:
        T = F_0(T)
        current_bit = "0"
    else:
        T = F_1(T)
        current_bit = "1"

(1, 1) = 2*(0, 1) - (-1, 1)   fixed:not
(1, 3) = 2*(-1, 3) - (-1, 1)   not:fixed
(1, 1) = 2*(-1, 3) - (-5, 3)   fixed:not
(1, 9) = 2*(-7, 9) - (-5, 3)   not:fixed
(1, 1) = 2*(-7, 9) - (-23, 9)   fixed:not


In [13]:
r = 3

T = N_i
current_bit = "1"
for i in range(5):
    val_1 = mrTupRootedValue(r, T)
    val_0 = mrTupRootedValue(0, T)
    val_n1 = mrTupRootedValue(-1, T)

    if current_bit == "0":
        status_0 = "not"
        status_1 = "fixed"
    else:
        status_0 = "fixed"
        status_1 = "not"
    
    print(f"{val_1} = {r+1}*{val_0} - {r}*{val_n1}   {status_0}:{status_1}")
    if i & 1 == 0:
        T = F_0(T)
        current_bit = "0"
    else:
        T = F_1(T)
        current_bit = "1"

(3, 1) = 4*(0, 1) - 3*(-1, 1)   fixed:not
(5, 3) = 4*(-1, 3) - 3*(-1, 1)   not:fixed
(11, 3) = 4*(-1, 3) - 3*(-5, 3)   fixed:not
(17, 9) = 4*(-7, 9) - 3*(-5, 3)   not:fixed
(41, 9) = 4*(-7, 9) - 3*(-23, 9)   fixed:not


In [8]:
# root 2 is a case of root 1, has period 2.
list(generationRootedIntegersForwardWithMeta(2, 8))

[(2, 0, 0, ''),
 (4, 1, 0, '1'),
 (1, 1, 1, '0'),
 (8, 2, 0, '11'),
 (2, 2, 1, '10'),
 (16, 3, 0, '111'),
 (5, 3, 1, '011'),
 (4, 3, 1, '110'),
 (1, 3, 2, '010'),
 (32, 4, 0, '1111'),
 (10, 4, 1, '1011'),
 (3, 4, 2, '0011'),
 (8, 4, 1, '1110'),
 (2, 4, 2, '1010'),
 (64, 5, 0, '11111'),
 (21, 5, 1, '01111'),
 (20, 5, 1, '11011'),
 (6, 5, 2, '10011'),
 (16, 5, 1, '11110'),
 (5, 5, 2, '01110'),
 (4, 5, 2, '11010'),
 (1, 5, 3, '01010'),
 (128, 6, 0, '111111'),
 (42, 6, 1, '101111'),
 (40, 6, 1, '111011'),
 (13, 6, 2, '011011'),
 (12, 6, 2, '110011'),
 (32, 6, 1, '111110'),
 (10, 6, 2, '101110'),
 (3, 6, 3, '001110'),
 (8, 6, 2, '111010'),
 (2, 6, 3, '101010'),
 (256, 7, 0, '1111111'),
 (85, 7, 1, '0111111'),
 (84, 7, 1, '1101111'),
 (80, 7, 1, '1111011'),
 (26, 7, 2, '1011011'),
 (24, 7, 2, '1110011'),
 (64, 7, 1, '1111110'),
 (21, 7, 2, '0111110'),
 (20, 7, 2, '1101110'),
 (6, 7, 3, '1001110'),
 (16, 7, 2, '1111010'),
 (5, 7, 3, '0111010'),
 (4, 7, 3, '1101010'),
 (1, 7, 4, '0101010'),
 (

# $V_{r}(\sigma) = (r+1) V_{0}(\sigma) - r V_{-1}(\sigma)$

#### TODO: Fix $F_0, F_1$ backwardness

To understand why any root $r > 3$ in your lattice $AX=Y$ eliminates the possibility of a cycle, we need to analyze the fixed-point mechanics of the backward substitution.

The Fixed-Point Equation for a CycleIn your lattice construction, starting from the root $r$ (the terminal value in $Y$), each backward step is defined by:

Even Step $$(F_0): $x_i = 2x_{i+1}$$

Odd Step $$(F_1): x_i = \frac{2x_{i+1} - 1}{3}$$ 

(only possible if $2x_{i+1} \equiv 1 \pmod 3$)

For a cycle of length $L$ with $j$ odd steps and $k = L-j$ even steps, the starting value $x_0$ is given by the affine composition:

$$
x_0 = \left( \frac{2^L}{3^j} \right) r - \beta
$$

where $\beta$ is the accumulated offset from the "$-1$" terms in the $F_1$ operations. For a cycle to exist, we must have $x_0 = r$. Solving for $r$:

$$
r = \left( \frac{2^L}{3^j} \right) r - \beta \implies r \left( 1 - \frac{2^L}{3^j} \right) = -\beta \implies r = \frac{\beta}{\frac{2^L}{3^j} - 1}
$$

3 is the dividing line and only contains integers of the form $3\cdot2^a$

Any root larger than 3 overwhelms the A matrix diagonal and eliminates the possibility of a cycle.


# All positive roots $3k$ are aperiodic in integers

The "Multiple of 3" Barrier ($r=3$)
 
If the root $r$ is a multiple of 3, the lattice enters a dead zone. 

$(2(3k)-1)/3$ is never an integer, therefore the only possible integer reverse operations in a lattice rooted at $3k$ are $n/2$ operations, so the only forward operation that gives an integer is $F_1$ and we have a lattice that only has integers on its upper boundary.

So all lattices of the form:

$V_{3k}(\sigma) = ((3k)+1) V_{0}(\sigma) - (3k) V_{-1}(\sigma)$

are aperiodic.

We can see this computationally.  The only integers in the lattice rooted at 3 are $(3,6,12,...)$ and the only integers in the lattice rooted at any 3k are $(3k, 2^1\cdot 3k, 2^2\cdot 3k, ...)$


In [10]:
list(generationRootedIntegersForwardWithMeta(3, 8))

[(3, 0, 0, ''),
 (6, 1, 0, '1'),
 (12, 2, 0, '11'),
 (24, 3, 0, '111'),
 (48, 4, 0, '1111'),
 (96, 5, 0, '11111'),
 (192, 6, 0, '111111'),
 (384, 7, 0, '1111111'),
 (768, 8, 0, '11111111')]

In [14]:
list(generationRootedIntegersForwardWithMeta(6, 8))

[(6, 0, 0, ''),
 (12, 1, 0, '1'),
 (24, 2, 0, '11'),
 (48, 3, 0, '111'),
 (96, 4, 0, '1111'),
 (192, 5, 0, '11111'),
 (384, 6, 0, '111111'),
 (768, 7, 0, '1111111'),
 (1536, 8, 0, '11111111')]

$$
V_{3k}(\sigma) = ((3k)+1) V_{0}(\sigma) - (3k) V_{-1}(\sigma)
$$

For all positive lattices > 3 we can write:
$$
\begin{aligned}
V_{3k+j}(\sigma)
  &= \underbrace{((3k)+1)V_{0}(\sigma) - (3k)V_{-1}(\sigma)}_{\text{dominant expanding}}
\\[6pt]
  &\quad + j \cdot \underbrace{(2V_{0}(\sigma) - V_{-1}(\sigma))}_{\text{period 2}}
\\[6pt]
  &\quad - j \cdot \underbrace{V_{0}(\sigma)}_{\text{period 1}}, 
  \qquad j \in (1,2)
\end{aligned}
$$


And it becomes clear that we cannot have a positive lattice with a cycle > 2.

In [11]:
# root 4 is aperiodic
list(generationRootedIntegersForwardWithMeta(4, 8))

[(4, 0, 0, ''),
 (8, 1, 0, '1'),
 (16, 2, 0, '11'),
 (5, 2, 1, '01'),
 (32, 3, 0, '111'),
 (10, 3, 1, '101'),
 (3, 3, 2, '001'),
 (64, 4, 0, '1111'),
 (21, 4, 1, '0111'),
 (20, 4, 1, '1101'),
 (6, 4, 2, '1001'),
 (128, 5, 0, '11111'),
 (42, 5, 1, '10111'),
 (40, 5, 1, '11101'),
 (13, 5, 2, '01101'),
 (12, 5, 2, '11001'),
 (256, 6, 0, '111111'),
 (85, 6, 1, '011111'),
 (84, 6, 1, '110111'),
 (80, 6, 1, '111101'),
 (26, 6, 2, '101101'),
 (24, 6, 2, '111001'),
 (512, 7, 0, '1111111'),
 (170, 7, 1, '1011111'),
 (168, 7, 1, '1110111'),
 (160, 7, 1, '1111101'),
 (53, 7, 2, '0111101'),
 (52, 7, 2, '1101101'),
 (17, 7, 3, '0101101'),
 (48, 7, 2, '1111001'),
 (1024, 8, 0, '11111111'),
 (341, 8, 1, '01111111'),
 (340, 8, 1, '11011111'),
 (113, 8, 2, '01011111'),
 (336, 8, 1, '11110111'),
 (320, 8, 1, '11111101'),
 (106, 8, 2, '10111101'),
 (35, 8, 3, '00111101'),
 (104, 8, 2, '11101101'),
 (34, 8, 3, '10101101'),
 (11, 8, 4, '00101101'),
 (96, 8, 2, '11111001')]

# Negative lattices

In [16]:
list(generationRootedIntegersForwardWithMeta(-3, 8))

[(-3, 0, 0, ''),
 (-6, 1, 0, '1'),
 (-12, 2, 0, '11'),
 (-24, 3, 0, '111'),
 (-48, 4, 0, '1111'),
 (-96, 5, 0, '11111'),
 (-192, 6, 0, '111111'),
 (-384, 7, 0, '1111111'),
 (-768, 8, 0, '11111111')]